In [31]:
import torch
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR
torch.cuda.is_available()

True

In [32]:
device = "cuda"

In [33]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset1 = datasets.MNIST('../data', train=True, download=True,
                    transform=transform)
dataset2 = datasets.MNIST('../data', train=False,
                    transform=transform)
example = dataset1[0]
len(example)

2

In [34]:
img, label = example
img.shape


torch.Size([1, 28, 28])

In [36]:
img.shape, label

(torch.Size([1, 28, 28]), 5)

In [5]:
args = argparse.Namespace(batch_size=64, test_batch_size=64, lr=0.01, gamma=0.7, seed=1, epochs=1, no_cuda=False, no_mps=False, log_interval=10, save_model=False, dry_run=False)
train_kwargs = {'batch_size': args.batch_size}
test_kwargs = {'batch_size': args.test_batch_size}
use_cuda = not args.no_cuda and torch.cuda.is_available()
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)


In [6]:
train_loader = torch.utils.data.DataLoader(dataset1,**train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset2, **test_kwargs)
for batch in train_loader:
    break
imgs, preds = batch
imgs.shape, preds.shape

(torch.Size([64, 1, 28, 28]), torch.Size([64]))

In [7]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output

In [8]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

In [9]:
model = Net().to(device)
optimizer = optim.Adadelta(model.parameters(), lr=args.lr)
scheduler = StepLR(optimizer, step_size=1, gamma=args.gamma)
for epoch in range(1, args.epochs + 1):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % args.log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if args.dry_run:
                break
    test(model, device, test_loader)
    scheduler.step()


Train Epoch: 1 [0/60000 (0%)]	Loss: 2.281838
Train Epoch: 1 [640/60000 (1%)]	Loss: 2.259301
Train Epoch: 1 [1280/60000 (2%)]	Loss: 2.244700
Train Epoch: 1 [1920/60000 (3%)]	Loss: 2.140908
Train Epoch: 1 [2560/60000 (4%)]	Loss: 2.086044
Train Epoch: 1 [3200/60000 (5%)]	Loss: 2.020277
Train Epoch: 1 [3840/60000 (6%)]	Loss: 1.928534
Train Epoch: 1 [4480/60000 (7%)]	Loss: 1.800162
Train Epoch: 1 [5120/60000 (9%)]	Loss: 1.661970
Train Epoch: 1 [5760/60000 (10%)]	Loss: 1.518022
Train Epoch: 1 [6400/60000 (11%)]	Loss: 1.416213
Train Epoch: 1 [7040/60000 (12%)]	Loss: 1.334159
Train Epoch: 1 [7680/60000 (13%)]	Loss: 1.432100
Train Epoch: 1 [8320/60000 (14%)]	Loss: 1.242771
Train Epoch: 1 [8960/60000 (15%)]	Loss: 1.276910
Train Epoch: 1 [9600/60000 (16%)]	Loss: 1.048678
Train Epoch: 1 [10240/60000 (17%)]	Loss: 0.971013
Train Epoch: 1 [10880/60000 (18%)]	Loss: 1.044301
Train Epoch: 1 [11520/60000 (19%)]	Loss: 0.846729
Train Epoch: 1 [12160/60000 (20%)]	Loss: 0.871627
Train Epoch: 1 [12800/60000 (

In [37]:
output = model(data)
loss = F.nll_loss(output, target)
loss

tensor(0.2469, device='cuda:0', grad_fn=<NllLossBackward0>)

In [38]:
F.nll_loss(output[:4], target[:4])

tensor(0.0803, device='cuda:0', grad_fn=<NllLossBackward0>)

In [40]:
sum = 0.0
for i, param in enumerate(model.parameters()):
    sum += torch.sum(param)
sum

tensor(158.4672, device='cuda:0', grad_fn=<AddBackward0>)

In [39]:
param

Parameter containing:
tensor([[[[ 0.3035,  0.1104,  0.3583],
          [ 0.1449, -0.2725, -0.0214],
          [-0.2352, -0.1152,  0.0478]]],


        [[[-0.3222,  0.1957,  0.2473],
          [-0.1672, -0.3284, -0.2469],
          [-0.2910, -0.2935,  0.0626]]],


        [[[ 0.3773,  0.1761,  0.1899],
          [-0.2804,  0.0990, -0.0992],
          [ 0.2228, -0.3193,  0.0950]]],


        [[[ 0.3122,  0.2166,  0.3459],
          [-0.2342, -0.0435,  0.1262],
          [-0.1962, -0.1521, -0.0101]]],


        [[[-0.2046,  0.2343,  0.2245],
          [-0.0473, -0.1435, -0.2822],
          [-0.3274, -0.2911, -0.2942]]],


        [[[-0.3072,  0.3305, -0.1294],
          [ 0.3599,  0.0599,  0.2772],
          [ 0.2210, -0.1625, -0.2741]]],


        [[[ 0.0820,  0.3252,  0.1641],
          [ 0.1773,  0.3705,  0.1375],
          [ 0.0804, -0.0590, -0.2231]]],


        [[[-0.1088,  0.2062, -0.0496],
          [ 0.0520, -0.0801, -0.1391],
          [ 0.1609,  0.1474,  0.2906]]],


        [[

In [20]:
param.grad is None

True

In [21]:
loss.backward()
param.grad is None


False

In [22]:
param.grad.shape

torch.Size([32, 1, 3, 3])

In [23]:
optimizer.zero_grad()
param.grad is None


True